# 02 - Aplicacion del modelo final a datos de test

Esta notebook carga el modelo final guardado por `01_train_final_model.ipynb` y replica las predicciones sobre el conjunto de test interno usando el umbral optimizado por F2 del modelo seleccionado por PR-AUC.

La consigna pide que este archivo permita aplicar el modelo final a los datos de test dados los datos de entrada y el modelo generado. Por eso no se entrena ningun modelo aca: solamente se reconstruyen las mismas features, se recrea la misma particion y se aplica `models/final_model.joblib`.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, fbeta_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_PATH = ROOT / "models" / "final_model.joblib"
OUTPUTS_DIR = ROOT / "outputs"

print("Root:", ROOT)
print("Dataset:", DATA_PATH)
print("Modelo:", MODEL_PATH)

Root: C:\Users\matut\OneDrive\Escritorio\Tp 1 Predictivo\Telco-Churn-Examen-Final
Dataset: C:\Users\matut\OneDrive\Escritorio\Tp 1 Predictivo\Telco-Churn-Examen-Final\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv
Modelo: C:\Users\matut\OneDrive\Escritorio\Tp 1 Predictivo\Telco-Churn-Examen-Final\models\final_model.joblib


## Carga de datos y modelo

Se carga el dataset original y el modelo final ya entrenado. Esta notebook depende de haber ejecutado primero la notebook de entrenamiento.

In [2]:
df_raw = pd.read_csv(DATA_PATH)
final_model = joblib.load(MODEL_PATH)

metrics_path = OUTPUTS_DIR / "model_metrics.json"
with metrics_path.open("r", encoding="utf-8") as f:
    saved_metrics = json.load(f)

best_threshold = float(saved_metrics["decision_threshold"])

print(df_raw.shape)
print("Umbral cargado:", best_threshold)
df_raw.head()

(7043, 21)
Umbral cargado: 0.31


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Feature engineering reproducible

Se aplica la misma transformacion de variables usada durante el entrenamiento. Mantener esta funcion igual en ambas notebooks es clave para que el modelo reciba las columnas esperadas.

In [3]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    data["TotalCharges_was_missing"] = data["TotalCharges"].isna().astype(int)
    data["TotalCharges"] = data["TotalCharges"].fillna(0)

    tenure_safe = data["tenure"].replace(0, np.nan)
    data["avg_charge_per_tenure"] = (data["TotalCharges"] / tenure_safe).replace([np.inf, -np.inf], np.nan).fillna(0)
    data["charges_gap"] = data["MonthlyCharges"] - data["avg_charge_per_tenure"]
    data["tenure_is_zero"] = (data["tenure"] == 0).astype(int)

    bins = [-1, 6, 12, 24, 48, np.inf]
    labels = ["0-6", "7-12", "13-24", "25-48", "49+"]
    data["tenure_group"] = pd.cut(data["tenure"], bins=bins, labels=labels).astype("object")

    data["is_month_to_month"] = (data["Contract"] == "Month-to-month").astype(int)
    data["has_fiber_optic"] = (data["InternetService"] == "Fiber optic").astype(int)
    data["is_electronic_check"] = (data["PaymentMethod"] == "Electronic check").astype(int)
    data["paperless_billing_flag"] = (data["PaperlessBilling"] == "Yes").astype(int)
    data["digital_payment_risk"] = (data["paperless_billing_flag"] & data["is_electronic_check"]).astype(int)

    support_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
    streaming_cols = ["StreamingTV", "StreamingMovies"]
    service_cols = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

    data["support_services_count"] = (data[support_cols] == "Yes").sum(axis=1)
    data["streaming_services_count"] = (data[streaming_cols] == "Yes").sum(axis=1)
    data["phone_or_addon_services_count"] = (data[service_cols] == "Yes").sum(axis=1)
    data["has_internet_service"] = (data["InternetService"] != "No").astype(int)
    data["total_services_count"] = data["phone_or_addon_services_count"] + data["has_internet_service"]
    data["has_support_services"] = (data["support_services_count"] > 0).astype(int)

    return data

## Reconstruccion del test

Como no hay un archivo de test externo separado, se replica exactamente la misma particion interna usada en entrenamiento:

- `test_size = 0.20`
- `stratify = Churn`
- `random_state = 42`

Esto permite reproducir el archivo de predicciones generado por la notebook de entrenamiento.

In [4]:
model_df = build_features(df_raw)
y = (model_df["Churn"] == "Yes").astype(int)
X = model_df.drop(columns=["Churn", "customerID"])

_, X_test, _, y_test, _, customer_test = train_test_split(
    X,
    y,
    model_df["customerID"],
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Test:", X_test.shape)
print("Churn test:", y_test.mean().round(4))

Test: (1409, 35)
Churn test: 0.2654


## Prediccion con el modelo guardado

Se generan probabilidades y etiquetas predichas usando el umbral optimizado por F2 en la notebook de entrenamiento. El archivo exportado queda en `outputs/test_predictions_from_saved_model.csv`.

In [5]:
y_proba = final_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= best_threshold).astype(int)

predictions = pd.DataFrame({
    "customerID": customer_test.values,
    "y_true": y_test.values,
    "churn_probability": y_proba,
    "decision_threshold": best_threshold,
    "churn_prediction": y_pred,
})
predictions["y_true_label"] = predictions["y_true"].map({0: "No", 1: "Yes"})
predictions["churn_prediction_label"] = predictions["churn_prediction"].map({0: "No", 1: "Yes"})

output_path = OUTPUTS_DIR / "test_predictions_from_saved_model.csv"
predictions.to_csv(output_path, index=False)

print("Predicciones guardadas en:", output_path)
predictions.head()

Predicciones guardadas en: C:\Users\matut\OneDrive\Escritorio\Tp 1 Predictivo\Telco-Churn-Examen-Final\outputs\test_predictions_from_saved_model.csv


,customerID,y_true,churn_probability,decision_threshold,churn_prediction,y_true_label,churn_prediction_label
0,4376-KFVRS,0,0.121948,0.31,0,No,No
1,2754-SDJRD,0,0.841516,0.31,1,No,Yes
2,9917-KWRBE,0,0.142365,0.31,0,No,No
3,0365-GXEZS,0,0.585634,0.31,1,No,Yes
4,9385-NXKDA,0,0.079332,0.31,0,No,No


## Verificacion de replicabilidad

Si existe el archivo generado en entrenamiento, se compara contra el archivo generado en esta notebook. La coincidencia confirma que el modelo guardado y la particion reproducen las mismas predicciones.

In [6]:
training_predictions_path = OUTPUTS_DIR / "test_predictions.csv"

if training_predictions_path.exists():
    original = pd.read_csv(training_predictions_path)
    replicated = pd.read_csv(output_path)
    same_predictions = original["churn_prediction"].equals(replicated["churn_prediction"])
    max_probability_difference = (original["churn_probability"] - replicated["churn_probability"]).abs().max()
    print("Mismas etiquetas predichas:", same_predictions)
    print("Maxima diferencia de probabilidad:", max_probability_difference)
else:
    print("No existe outputs/test_predictions.csv para comparar. Ejecutar primero 01_train_final_model.ipynb.")

Mismas etiquetas predichas: True
Maxima diferencia de probabilidad: 0.0


## Metricas sobre test

Se recalculan metricas para documentar el resultado del modelo aplicado desde disco.

In [7]:
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "f2": fbeta_score(y_test, y_pred, beta=2, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

pd.DataFrame([metrics]).T.rename(columns={0: "valor"})

,valor
accuracy,0.662881
precision,0.435832
recall,0.917112
f1,0.590870
f2,0.751205
roc_auc,0.844607


## Cierre

Esta notebook no modifica el entrenamiento. Su funcion es aplicar el modelo final guardado y demostrar que las predicciones de test pueden reproducirse desde los datos de entrada y el artefacto `final_model.joblib`.